# Week 3 · Class 2 — Vector Search at Scale: FAISS

Your Class 1 NumPy search compares the query against **every** vector. That's fine for 200
chunks and hopeless for 100 million. Today: FAISS indexes — exact (`Flat`), clustered (`IVF`),
and graph-based (`HNSW`) — benchmarked live on your own runtime, with **recall@k** as the
quality yardstick.

## Before you start

Standalone, no API key needed. Section 2 builds a small real corpus plus a 100K synthetic one
so the benchmarks show real differences even in Colab.

## Section 1 — Install packages

In [ ]:
!pip install -q faiss-cpu sentence-transformers pandas matplotlib

In [ ]:
import faiss
import numpy as np
print("FAISS version:", faiss.__version__)

## Section 2 — Two corpora: real and big

- **Real**: a small chunk corpus embedded with MiniLM (use your Week 2 `chunks.json` if you have it)
- **Big**: 100,000 random 384-D vectors — meaningless text-wise, perfect for timing

> Random vectors are a standard benchmarking trick: index speed depends on N and dimension,
> not on whether the numbers came from Shakespeare.

In [ ]:
import json, os
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
d = 384

corpus = [
    "Automobile repair pricing starts at 80 pounds per hour at our garage.",
    "The 2026 FIFA World Cup will be hosted by the USA, Canada, and Mexico.",
    "Python 3.13 introduced an experimental free-threaded mode without the GIL.",
    "Our refund policy allows returns within 30 days with a valid receipt.",
    "The goalkeeper made three incredible saves in the penalty shootout.",
    "To install packages, run pip install followed by the package name.",
    "Deep dish pizza originated in Chicago in the 1940s.",
    "Student visas require proof of enrollment and financial support documents.",
]
sources = ["garage.txt", "wiki", "python-blog", "policy.pdf",
           "match-report", "python-blog", "food-blog", "policy.pdf"]

# Prefer your Week 2 chunks if present
if os.path.exists("data/chunks.json"):
    raw = json.load(open("data/chunks.json"))
    corpus = [c if isinstance(c, str) else c["text"] for c in raw]
    sources = ["chunks.json"] * len(corpus)
    print(f"Using YOUR {len(corpus)} Week 2 chunks")

real_embs = model.encode(corpus, normalize_embeddings=True).astype("float32")
print("real corpus:", real_embs.shape)

# 100K synthetic vectors for benchmarking (normalized, like real embeddings)
rng = np.random.default_rng(42)
big = rng.normal(size=(100_000, d)).astype("float32")
faiss.normalize_L2(big)
big_queries = rng.normal(size=(100, d)).astype("float32")
faiss.normalize_L2(big_queries)
print("benchmark corpus:", big.shape)

## Section 3 — The Flat index: exact search, C++ speed

`IndexFlatIP` computes the inner product against everything — same math as your NumPy loop.
Because we **L2-normalized** the vectors, inner product **equals cosine similarity**.

In [ ]:
index = faiss.IndexFlatIP(d)
index.add(real_embs)
print("vectors stored:", index.ntotal)

q = model.encode(["how much to fix my car?"], normalize_embeddings=True).astype("float32")
scores, ids = index.search(q, k=3)
print("scores:", np.round(scores[0], 3))
print("ids:   ", ids[0])

**FAISS returns positions, not text.** Keep a parallel `chunk_store` list — position `i`
in the store is vector `i` in the index. (Class 3's vector database does this bookkeeping for you.)

In [ ]:
chunk_store = [{"text": t, "source": s} for t, s in zip(corpus, sources)]

def faiss_search(index, query: str, k: int = 3):
    qv = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    return [
        (float(score), chunk_store[i])
        for score, i in zip(scores[0], ids[0]) if i != -1
    ]

for score, hit in faiss_search(index, "how much to fix my car?"):
    print(f"{score:.3f}  [{hit['source']}]  {hit['text'][:80]}")

### 🔹 Mini-exercise (3 min)

Verify FAISS agrees with Class 1: compute the same top-3 with a plain NumPy dot product over
`real_embs` and confirm the ids match. If they don't, something is un-normalized — find it.

In [ ]:
# TODO: NumPy top-3 for the same query, compare with ids[0] above
qv = model.encode("how much to fix my car?", normalize_embeddings=True)
numpy_top3 = ...   # np.argsort(...)[...]
print("NumPy top-3:", numpy_top3)

## Section 4 — The scale wall, measured

How long does exact search take on 100K vectors? And what happens as N grows?

In [ ]:
import time

def bench(index, queries, k=10, repeats=3):
    """Median seconds for one full query batch."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        index.search(queries, k)
        times.append(time.perf_counter() - t0)
    return sorted(times)[len(times) // 2]

flat_big = faiss.IndexFlatIP(d)
flat_big.add(big)

t_flat = bench(flat_big, big_queries)
print(f"Flat (exact), 100 queries over 100K vectors: {t_flat*1000:.1f} ms")
print(f"→ per query: {t_flat*10:.2f} ms — now imagine 100M vectors (1000× more)…")

# ground truth for recall measurements later
true_scores, true_ids = flat_big.search(big_queries, 10)

## Section 5 — IVF: cluster first, search less

**I**nverted **F**ile index: k-means clusters the corpus into `nlist` cells at build time.
At query time, only the `nprobe` nearest cells are scanned — most vectors are never touched.

- `nprobe = 1` → fastest, risk of missing true neighbors that fell in a skipped cell
- `nprobe = nlist` → exact again, but slow

`nprobe` is your **quality dial**.

In [ ]:
nlist = 100
quantizer = faiss.IndexFlatIP(d)
ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

ivf.train(big)      # k-means over the corpus → 100 centroids
ivf.add(big)
print("trained:", ivf.is_trained, "| vectors:", ivf.ntotal)

In [ ]:
def recall_at_k(found_ids, truth_ids, k=10):
    return float(np.mean([
        len(set(f[:k]) & set(t[:k])) / k
        for f, t in zip(found_ids, truth_ids)
    ]))

results = []
for nprobe in [1, 2, 5, 10, 20, 50]:
    ivf.nprobe = nprobe
    t = bench(ivf, big_queries)
    _, ids = ivf.search(big_queries, 10)
    r = recall_at_k(ids, true_ids)
    results.append({"index": f"IVF nprobe={nprobe}", "ms/100q": t * 1000,
                    "speedup": t_flat / t, "recall@10": r})
    print(f"nprobe={nprobe:>3}  {t*1000:7.1f} ms   {t_flat/t:5.1f}x faster   recall@10={r:.3f}")

Read the trade-off directly off your own machine: tiny `nprobe` → huge speedup, real
recall loss. Around `nprobe≈10` (10% of cells) recall is usually ~0.95+ while still ~10× faster.

### 🔹 Mini-exercise (5 min)

Find the **smallest** `nprobe` that reaches **recall@10 ≥ 0.95** on your runtime.
Then answer in a comment: why is that number not the same on everyone's machine/dataset?

## Section 6 — HNSW: a graph you can ski down

**H**ierarchical **N**avigable **S**mall **W**orld: every vector is a node linked to ~M
neighbors, with sparse "express" layers on top (like a skip list). A query enters at the top and
greedily hops closer.

- No training step; inserts are slower and RAM is higher (the graph)
- **Best query-time recall/speed trade** — it's what most vector DBs (incl. Qdrant) run inside

In [ ]:
hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)  # M=32 links/node
print("building graph (slowest build of the three)…")
hnsw.add(big)

for ef in [16, 64, 128]:
    hnsw.hnsw.efSearch = ef
    t = bench(hnsw, big_queries)
    _, ids = hnsw.search(big_queries, 10)
    r = recall_at_k(ids, true_ids)
    results.append({"index": f"HNSW ef={ef}", "ms/100q": t * 1000,
                    "speedup": t_flat / t, "recall@10": r})
    print(f"efSearch={ef:>4}  {t*1000:7.1f} ms   {t_flat/t:5.1f}x faster   recall@10={r:.3f}")

## Section 7 — The scoreboard

In [ ]:
import pandas as pd

results.insert(0, {"index": "Flat (exact)", "ms/100q": t_flat * 1000,
                   "speedup": 1.0, "recall@10": 1.0})
df = pd.DataFrame(results).round({"ms/100q": 1, "speedup": 1, "recall@10": 3})
df

In [ ]:
import matplotlib.pyplot as plt

ann = df[df["index"] != "Flat (exact)"]
plt.figure(figsize=(8, 5), facecolor="white")
plt.scatter(ann["speedup"], ann["recall@10"], s=90, c="#8b5cf6")
for _, row in ann.iterrows():
    plt.annotate(row["index"], (row["speedup"], row["recall@10"]),
                 fontsize=8, xytext=(5, 5), textcoords="offset points")
plt.axhline(0.95, color="#22d3ee", ls="--", lw=1, label="0.95 recall target")
plt.xlabel("speedup vs exact (×)"); plt.ylabel("recall@10")
plt.title("The ANN trade-off, measured on YOUR runtime")
plt.legend(); plt.tight_layout(); plt.show()

**The RAG argument:** your LLM reads the top 3–5 chunks. If an ANN index returns 9.5 of
the true top-10 on average, the final answer almost never changes — but every query got 10–50×
cheaper. That's why production RAG is built on ANN.

## Section 8 — Persistence: embed once, load forever

Embedding is the expensive step. Never redo it — save the index **and** the chunk store
(they must stay in the same order).

In [ ]:
index_real = faiss.IndexFlatIP(d)
index_real.add(real_embs)

faiss.write_index(index_real, "week3_chunks.faiss")
json.dump(chunk_store, open("chunk_store.json", "w"), indent=2)
print("saved: week3_chunks.faiss + chunk_store.json")

# --- fresh session simulation ---
index2 = faiss.read_index("week3_chunks.faiss")
store2 = json.load(open("chunk_store.json"))
print("reloaded:", index2.ntotal, "vectors,", len(store2), "chunks")

for score, hit in faiss_search(index2, "what is the returns policy?"):
    print(f"{score:.3f}  [{hit['source']}]  {hit['text'][:80]}")

> **Colab tip:** the runtime's disk vanishes when the session ends — download the two files
> or copy them to Google Drive if you want them to survive.

## Section 9 — Choosing an index (cheat sheet)

| Situation | Use | Why |
|-----------|-----|-----|
| < 100K vectors (your project!) | `IndexFlatIP` | exact, instant, zero tuning |
| 100K – 10M, RAM fine | `HNSW` | best query-time recall/speed |
| 10M+, RAM tight | `IVF` (+ PQ compression) | tunable memory & speed |
| Need filters, CRUD, a server | vector DB (Class 3!) | FAISS is a library, not a database |

**Rule of thumb: start Flat, graduate when latency hurts.**

## Section 10 — Homework

1. **Index your Week 2 data** — build and save a Flat index over your real chunks + book
   dataset (not the fallback corpus). Keep `week3_chunks.faiss` + `chunk_store.json` for Class 3.
2. **The knee plot** — extend the nprobe sweep to `{1,2,5,10,20,50,100}` and plot recall vs
   time. Mark the "knee" where extra nprobe stops paying.
3. **Break IVF** — with `nprobe=1`, find a query vector whose true nearest neighbor is NOT
   returned. (Hint: query near a boundary between two clusters.) Explain in two sentences.
4. **Stretch** — try `faiss.IndexIVFPQ(quantizer, d, nlist, 8, 8)`: how much smaller is the
   index in memory, and what did recall@10 cost?

## Checklist before Class 3

- [ ] I can explain nprobe and efSearch as quality dials
- [ ] I measured recall@10 myself and know what 0.95 means
- [ ] My real chunks are saved as index + store files
- [ ] I know why FAISS alone isn't enough for the agent — that's where Qdrant comes in